# SQL query from table names - Continued

In [1]:
from openai import OpenAI
import os
from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv())

OPENAI_API_KEY  = os.getenv('OPENAI_API_KEY')

## The old Prompt

In [2]:
#The old prompt
old_context = [ {'role':'system', 'content':"""
you are a bot to assist in create SQL commands, all your answers should start with \
this is your SQL, and after that an SQL that can do what the user request. \
Your Database is composed by a SQL database with some tables. \
Try to maintain the SQL order simple.
Put the SQL command in white letters with a black background, and just after \
a simple and concise text explaining how it works.
If the user ask for something that can not be solved with an SQL Order \
just answer something nice and simple, maximum 10 words, asking him for something that \
can be solved with SQL.
"""} ]

old_context.append( {'role':'system', 'content':"""
first table:
{
  "tableName": "employees",
  "fields": [
    {
      "nombre": "ID_usr",
      "tipo": "int"
    },
    {
      "nombre": "name",
      "tipo": "varchar"
    }
  ]
}
"""
})

old_context.append( {'role':'system', 'content':"""
second table:
{
  "tableName": "salary",
  "fields": [
    {
      "nombre": "ID_usr",
      "type": "int"
    },
    {
      "name": "year",
      "type": "date"
    },
    {
      "name": "salary",
      "type": "float"
    }
  ]
}
"""
})

old_context.append( {'role':'system', 'content':"""
third table:
{
  "tablename": "studies",
  "fields": [
    {
      "name": "ID",
      "type": "int"
    },
    {
      "name": "ID_usr",
      "type": "int"
    },
    {
      "name": "educational_level",
      "type": "int"
    },
    {
      "name": "Institution",
      "type": "varchar"
    },
    {
      "name": "Years",
      "type": "date"
    }
    {
      "name": "Speciality",
      "type": "varchar"
    }
  ]
}
"""
})

## New Prompt.
We are going to improve it following the instructions of a Paper from the Ohaio University: [How to Prompt LLMs for Text-to-SQL: A Study in Zero-shot, Single-domain, and Cross-domain Settings](https://arxiv.org/abs/2305.11853). I recommend you read that paper.

For each table, we will define the structure using the same syntax as in a SQL create table command, and add the sample rows of the content.

Finally, at the end of the prompt, we'll include some example queries with the SQL that the model should generate. This technique is called Few-Shot Samples, in which we provide the prompt with some examples to assist it in generating the correct SQL.


In [10]:
context = [ {'role':'system', 'content':"""
CREATE TABLE employees (
    ID_Usr INT PRIMARY KEY,
    name VARCHAR(100)
);
-- Sample rows:
-- ID_Usr | name
-- 1      | Alice Johnson
-- 2      | Bob Smith
-- 3      | Carol White

CREATE TABLE salary (
    ID_Usr INT,
    year DATE,
    salary FLOAT,
    FOREIGN KEY (ID_Usr) REFERENCES employees(ID_Usr)
);
-- Sample rows:
-- ID_Usr | year       | salary
-- 1      | 2023-01-01 | 85000.00
-- 2      | 2023-01-01 | 72000.00
-- 3      | 2023-01-01 | 95000.00

CREATE TABLE studies (
    ID INT PRIMARY KEY,
    ID_Usr INT,
    educational_level INT,
    Institution VARCHAR(200),
    Years DATE,
    Speciality VARCHAR(200),
    FOREIGN KEY (ID_Usr) REFERENCES employees(ID_Usr)
);
-- Sample rows:
-- ID | ID_Usr | educational_level | Institution      | Years      | Speciality
-- 1  | 1      | 3                 | MIT              | 2015-06-01 | Computer Science
-- 2  | 2      | 2                 | State University | 2012-06-01 | Business Admin
-- 3  | 3      | 4                 | Stanford         | 2018-06-01 | Data Science
"""} ]



In [11]:
#FEW SHOT SAMPLES
context.append( {'role':'system', 'content':"""
 -- Maintain the SQL order simple and efficient as you can, using valid SQL Lite, answer the following questions for the table provided above.
WRITE IN YOUR CONTEXT QUERIES HERE
Q: How many employees are there?
A: SELECT COUNT(*) FROM employees;

Q: What is the total salary paid in 2023?
A: SELECT SUM(salary) FROM salary WHERE year = '2023-01-01';

Q: List all employees and their speciality.
A: SELECT e.name, st.Speciality FROM employees e JOIN studies st ON e.ID_Usr = st.ID_Usr;
"""
})

In [12]:
#Functio to call the model.
def return_CCRMSQL(user_message, context):
    client = OpenAI(
    # This is the default and can be omitted
    api_key=OPENAI_API_KEY,
)

    newcontext = context.copy()
    newcontext.append({'role':'user', 'content':"question: " + user_message})

    response = client.chat.completions.create(
            model="gpt-3.5-turbo",
            messages=newcontext,
            temperature=0,
        )

    return (response.choices[0].message.content)

## NL2SQL Samples
We're going to review some examples generated with the old prompt and others with the new prompt.

In [16]:
#new
context_user = context.copy()
print(return_CCRMSQL("Which employee has the highest salary?", context_user))

```sql
SELECT e.name
FROM employees e
JOIN salary s ON e.ID_Usr = s.ID_Usr
WHERE s.salary = (SELECT MAX(salary) FROM salary);
```


In [17]:
#old
old_context_user = old_context.copy()
print(return_CCRMSQL("Which employee has the highest salary?", old_context_user))

This is your SQL:
```sql
SELECT e.name
FROM employees e
JOIN salary s ON e.ID_usr = s.ID_usr
ORDER BY s.salary DESC
LIMIT 1;
```

This SQL query selects the name of the employee with the highest salary by joining the "employees" and "salary" tables on the employee ID. It then orders the result by salary in descending order and limits the output to only one row, which represents the employee with the highest salary.


In [15]:
#new
print(return_CCRMSQL("Which institution has the highest average salary among its graduates?", context_user))

SELECT e.name, st.Speciality FROM employees e JOIN studies st ON e.ID_Usr = st.ID_Usr;


In [18]:
#old
print(return_CCRMSQL("Which institution has the highest average salary among its graduates?", old_context_user))

This is your SQL:
```sql
SELECT Institution, AVG(salary) AS avg_salary
FROM studies
JOIN employees ON studies.ID_usr = employees.ID_usr
JOIN salary ON studies.ID_usr = salary.ID_usr
GROUP BY Institution
ORDER BY avg_salary DESC
LIMIT 1;
```

This SQL query joins the tables `studies`, `employees`, and `salary` on the `ID_usr` column to calculate the average salary for each institution. It then selects the institution with the highest average salary among its graduates by ordering the results in descending order and limiting the output to the top result.


# Exercise
 - Complete the prompts similar to what we did in class. 
     - Try at least 3 versions
     - Be creative
 - Write a one page report summarizing your findings.
     - Were there variations that didn't work well? i.e., where GPT either hallucinated or wrong.
     - What did you learn?

In [19]:
#new
print(return_CCRMSQL("Which institution has the highest average salary among its graduates?", context_user))

```sql
SELECT st.Institution, AVG(sa.salary) AS avg_salary
FROM employees e
JOIN studies st ON e.ID_Usr = st.ID_Usr
JOIN salary sa ON e.ID_Usr = sa.ID_Usr
GROUP BY st.Institution
ORDER BY avg_salary DESC
LIMIT 1;
```


In [20]:
#old
print(return_CCRMSQL("Which institution has the highest average salary among its graduates?", old_context_user))

This is your SQL:
```sql
SELECT s.Institution, AVG(sa.salary) AS avg_salary
FROM studies s
JOIN employees e ON s.ID_usr = e.ID_usr
JOIN salary sa ON s.ID_usr = sa.ID_usr
GROUP BY s.Institution
ORDER BY avg_salary DESC
LIMIT 1;
```

This SQL query joins the tables `studies`, `employees`, and `salary` to calculate the average salary for each institution where graduates studied. It then selects the institution with the highest average salary among its graduates.


## Findings Report: Old Prompt vs. New Prompt for NL-to-SQL

2. **Few-shot examples enforce consistent output** — providing 2–3 example Q&A pairs dramatically reduces format variation.
3. **Sample rows reduce hallucination** — when the model can see actual data values, it's less likely to invent columns or misidentify types.